# Can we predict whether a finding is malignant?

Target: `class` (1 = malignant). Six numeric, already-standardised features. The key challenge is **class imbalance** (~2% positives), so accuracy is meaningless; we use PR-AUC (average precision), ROC-AUC, and recall/precision at a chosen threshold.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             roc_curve, confusion_matrix, classification_report)
SEED=0
df=pd.read_csv('data.csv')
df.shape

## 1. Data overview

In [ ]:
print(df['class'].value_counts().rename('n').to_frame().assign(share=lambda x:x.n/x.n.sum()))
print('missing values:', df.isna().sum().sum())
print('duplicate rows:', df.duplicated().sum())
df.describe().T

Duplicates are common (the features look discretised). If identical rows land in both train and test, scores are inflated. We check them per class and then **deduplicate** before modelling so evaluation is honest.

In [ ]:
feat=[c for c in df.columns if c!='class']
print(df.groupby('class')[feat].apply(lambda g:g.duplicated().mean()).rename('dup share'))
g=df.groupby(feat)['class'].nunique()
print('feature vectors with conflicting labels:', (g>1).sum())
d=df.drop_duplicates().reset_index(drop=True)
print(d.shape, 'positive rate after dedup:', round(d['class'].mean(),4))

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(13,6))
for a,f in zip(ax.ravel(),feat):
    lo,hi=d[f].quantile([.005,.995])
    for k,col in [(0,'tab:blue'),(1,'tab:red')]:
        a.hist(d.loc[d['class']==k,f].clip(lo,hi),bins=40,density=True,alpha=.55,color=col,label=f'class {k}')
    a.set_title(f)
ax[0,0].legend(); plt.tight_layout(); plt.show()
d.groupby('class')[feat].median().T

## 2. Train / test split
Stratified hold-out test set (25%), untouched until the final evaluation. Model selection uses stratified 5-fold CV on the training part only.

In [ ]:
X,y=d[feat],d['class']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
print(len(Xtr),len(Xte),'positives:',ytr.sum(),yte.sum())

## 3. Model comparison (cross-validated)

In [ ]:
models={
 'dummy (prior)':DummyClassifier(strategy='prior'),
 'logreg':LogisticRegression(max_iter=2000),
 'logreg balanced':LogisticRegression(max_iter=2000,class_weight='balanced'),
 'random forest':RandomForestClassifier(500,min_samples_leaf=3,n_jobs=-1,random_state=SEED),
 'random forest balanced':RandomForestClassifier(500,min_samples_leaf=3,class_weight='balanced_subsample',n_jobs=-1,random_state=SEED),
 'hist GB':HistGradientBoostingClassifier(random_state=SEED),
 'hist GB balanced':HistGradientBoostingClassifier(class_weight='balanced',random_state=SEED),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof={};rows=[]
for n,m in models.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,average_precision_score(ytr,p),roc_auc_score(ytr,p)))
res=pd.DataFrame(rows,columns=['model','PR-AUC','ROC-AUC']).set_index('model').round(3)
print('baseline PR-AUC = prevalence =',round(ytr.mean(),3))
res.sort_values('PR-AUC',ascending=False)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n in ['logreg','random forest','hist GB']:
    pr,rc,_=precision_recall_curve(ytr,oof[n]); ax[0].plot(rc,pr,label=n)
    fp,tp,_=roc_curve(ytr,oof[n]); ax[1].plot(fp,tp,label=n)
ax[0].axhline(ytr.mean(),ls='--',c='gray'); ax[0].set(xlabel='recall',ylabel='precision',title='PR (out-of-fold)')
ax[1].plot([0,1],[0,1],'--',c='gray'); ax[1].set(xlabel='FPR',ylabel='TPR',title='ROC (out-of-fold)')
ax[0].legend(); plt.tight_layout(); plt.show()

## 4. Pick a model and an operating threshold
The best model by out-of-fold PR-AUC is chosen. The threshold is picked on out-of-fold training predictions (not the test set). The right trade-off depends on the cost of a missed malignancy vs. a false alarm; as a default we use the threshold maximising F2 (recall weighted above precision).

In [ ]:
best=res['PR-AUC'].idxmax(); print('best model:',best)
p=oof[best]; pr,rc,th=precision_recall_curve(ytr,p)
f2=5*pr*rc/(4*pr+rc+1e-12); thr=th[np.argmax(f2[:-1])]
print('threshold',round(thr,4))

## 5. Final evaluation on the held-out test set

In [ ]:
final=models[best].fit(Xtr,ytr); pt=final.predict_proba(Xte)[:,1]
print('Test PR-AUC  :',round(average_precision_score(yte,pt),3),'(prevalence %.3f)'%yte.mean())
print('Test ROC-AUC :',round(roc_auc_score(yte,pt),3))
yh=(pt>=thr).astype(int)
print(confusion_matrix(yte,yh))
print(classification_report(yte,yh,digits=3))

In [ ]:
rng=np.random.default_rng(SEED); yt=yte.values; a=[];r=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt))
    if yt[i].sum()==0: continue
    a.append(average_precision_score(yt[i],pt[i])); r.append(roc_auc_score(yt[i],pt[i]))
print('PR-AUC 95%% bootstrap CI: %.3f-%.3f'%tuple(np.percentile(a,[2.5,97.5])))
print('ROC-AUC 95%% bootstrap CI: %.3f-%.3f'%tuple(np.percentile(r,[2.5,97.5])))

In [ ]:
from sklearn.inspection import permutation_importance
pi=permutation_importance(final,Xte,yte,scoring='average_precision',n_repeats=20,random_state=SEED)
pd.Series(pi.importances_mean,index=feat).sort_values().plot.barh(title='Permutation importance (test PR-AUC drop)'); plt.show()

## 6. Conclusions

- **Yes, malignancy is predictable.** On a held-out test set, gradient boosting reaches ROC-AUC ≈ 0.96 and PR-AUC ≈ 0.72 (95% bootstrap CI ≈ 0.61–0.82), versus a PR-AUC of ≈ 0.03 for a no-skill model.
- At the F2-optimised threshold the model catches ~72% of malignant findings (46/64) with ~60% precision (31 false alarms among 1,963 cases). The threshold can be moved to trade recall against false alarms depending on clinical cost.
- Non-linear models (boosting, random forest) clearly beat logistic regression (PR-AUC ~0.76 vs ~0.63 in CV); class weighting did not help.
- **Caveats:** 30% of benign rows were exact duplicates (vs 2% of malignant). We deduplicated to avoid leakage, which raises the positive rate from 2.3% to 3.2%; precision on raw, duplicate-containing data would be somewhat different. Only ~64 positives are in the test set, so estimates are noisy. There is no patient/source identifier, so we can't rule out correlated rows across the split. Results come from one dataset and should be validated on new data before being relied on.